# Day 073 — Solution: Podcast Generator

In [ ]:
_PODCAST_SRC = '"""podcast_generator.py — Day 073: Text-to-Speech Deep Dive.\n\nGenerates podcast-style audio using Microsoft Edge TTS (free, local API).\nExtends the Edge TTS introduced on Day 1.\n\nSetup:\n    pip install edge-tts\n\nFor Jupyter notebooks (avoids "event loop already running" error):\n    pip install nest_asyncio\n    import nest_asyncio; nest_asyncio.apply()\n\nUsage:\n    from podcast_generator import PodcastGenerator\n\n    # Testing — no network needed\n    mock_tts = lambda text, **kw: b"AUDIO:" + text[:10].encode()\n    gen = PodcastGenerator(tts_fn=mock_tts)\n    parts = gen.build([\n        {"text": "Welcome to the show.", "voice_key": "host"},\n        {"text": "Thanks for having me!", "voice_key": "guest"},\n    ])\n    print(len(parts), parts[0][:12])   # 2  b\'AUDIO:Welcom\'\n\n    # Real generation (requires network)\n    gen = PodcastGenerator()\n    parts = gen.build([{"text": "Hello world.", "voice_key": "host"}])\n    open("out.mp3", "wb").write(parts[0])\n"""\nimport asyncio\nfrom pathlib import Path\nfrom typing import Callable, Optional\n\n# Common Edge TTS voices — subset for reference\nCOMMON_VOICES = [\n    {"ShortName": "en-US-AriaNeural",    "Gender": "Female", "Locale": "en-US"},\n    {"ShortName": "en-US-GuyNeural",     "Gender": "Male",   "Locale": "en-US"},\n    {"ShortName": "en-GB-LibbyNeural",   "Gender": "Female", "Locale": "en-GB"},\n    {"ShortName": "en-AU-NatashaNeural", "Gender": "Female", "Locale": "en-AU"},\n    {"ShortName": "fr-FR-DeniseNeural",  "Gender": "Female", "Locale": "fr-FR"},\n    {"ShortName": "de-DE-KatjaNeural",   "Gender": "Female", "Locale": "de-DE"},\n    {"ShortName": "es-ES-ElviraNeural",  "Gender": "Female", "Locale": "es-ES"},\n    {"ShortName": "ja-JP-NanamiNeural",  "Gender": "Female", "Locale": "ja-JP"},\n]\n\n# Role → voice name mapping used by PodcastGenerator\nDEFAULT_VOICE_MAP = {\n    "host":    "en-US-AriaNeural",\n    "guest":   "en-US-GuyNeural",\n    "narrator":"en-GB-LibbyNeural",\n}\n\n\ndef build_prosody_ssml(text: str, rate: str = "+0%",\n                        pitch: str = "+0Hz", volume: str = "+0%") -> str:\n    """Wrap text in SSML prosody tags for rate/pitch/volume control.\n\n    Returns a full SSML document string ready to pass to Edge TTS as text.\n    """\n    return (\n        \'<speak version="1.0" \'\n        \'xmlns="http://www.w3.org/2001/10/synthesis" xml:lang="en-US">\'\n        f\'<prosody rate="{rate}" pitch="{pitch}" volume="{volume}">\'\n        f"{text}"\n        "</prosody></speak>"\n    )\n\n\ndef select_voice(voices: list, locale: str = "en-US",\n                  gender: Optional[str] = None) -> Optional[dict]:\n    """Find the first voice matching locale (and optionally gender).\n\n    Args:\n        voices: list of voice dicts (ShortName, Gender, Locale)\n        locale: exact Locale string to match (e.g. \'en-US\', \'fr-FR\')\n        gender: \'Female\' or \'Male\' (case-insensitive); None = any\n    Returns:\n        First matching voice dict, or None if not found\n    """\n    for v in voices:\n        if v.get("Locale") != locale:\n            continue\n        if gender is not None and v.get("Gender", "").lower() != gender.lower():\n            continue\n        return v\n    return None\n\n\ndef synthesize(text: str, voice: str = "en-US-AriaNeural",\n               tts_fn: Optional[Callable] = None,\n               rate: str = "+0%", pitch: str = "+0Hz") -> bytes:\n    """Synthesize text to audio bytes using Edge TTS.\n\n    Args:\n        text:    Text or SSML string to synthesize\n        voice:   Edge TTS voice ShortName\n        tts_fn:  callable(text, voice, rate, pitch) -> bytes for testing\n        rate:    Speed adjustment e.g. \'+10%\', \'-20%\', \'+0%\'\n        pitch:   Pitch adjustment e.g. \'+5Hz\', \'-2Hz\', \'+0Hz\'\n    Returns:\n        Audio bytes (MP3 format when using real Edge TTS)\n    Note:\n        In Jupyter: pip install nest_asyncio; nest_asyncio.apply() before use.\n    """\n    if tts_fn is not None:\n        return tts_fn(text, voice=voice, rate=rate, pitch=pitch)\n    import edge_tts\n\n    async def _run() -> bytes:\n        communicate = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)\n        chunks: list[bytes] = []\n        async for chunk in communicate.stream():\n            if chunk["type"] == "audio":\n                chunks.append(chunk["data"])\n        return b"".join(chunks)\n\n    return asyncio.run(_run())\n\n\ndef synthesize_segments(segments: list,\n                         tts_fn: Optional[Callable] = None) -> list:\n    """Synthesize a list of segment dicts to audio bytes.\n\n    Each segment dict: {text, voice?, rate?, pitch?}\n    Returns list of bytes, one per segment.\n    """\n    out = []\n    for seg in segments:\n        audio = synthesize(\n            seg["text"],\n            voice=seg.get("voice", "en-US-AriaNeural"),\n            tts_fn=tts_fn,\n            rate=seg.get("rate", "+0%"),\n            pitch=seg.get("pitch", "+0Hz"),\n        )\n        out.append(audio)\n    return out\n\n\nclass PodcastGenerator:\n    """Generate podcast-style audio from a script using Edge TTS.\n\n    Inject tts_fn for testing without a network connection::\n\n        mock_tts = lambda text, **kw: b"AUDIO:" + text[:10].encode()\n        gen = PodcastGenerator(tts_fn=mock_tts)\n    """\n\n    def __init__(self, voice_map: Optional[dict] = None,\n                 tts_fn: Optional[Callable] = None) -> None:\n        self._voice_map = voice_map if voice_map is not None else DEFAULT_VOICE_MAP\n        self._tts_fn    = tts_fn\n\n    def synthesize_segment(self, text: str, voice_key: str = "host",\n                            rate: str = "+0%", pitch: str = "+0Hz") -> bytes:\n        """Synthesize one script segment. Returns audio bytes."""\n        voice = self._voice_map.get(voice_key, DEFAULT_VOICE_MAP["host"])\n        return synthesize(text, voice=voice, tts_fn=self._tts_fn,\n                          rate=rate, pitch=pitch)\n\n    def build(self, script: list) -> list:\n        """Synthesize all script entries. Returns list of bytes.\n\n        Each script entry: {text, voice_key?, rate?, pitch?}\n        """\n        return [\n            self.synthesize_segment(\n                entry["text"],\n                voice_key=entry.get("voice_key", "host"),\n                rate=entry.get("rate", "+0%"),\n                pitch=entry.get("pitch", "+0Hz"),\n            )\n            for entry in script\n        ]\n\n    def save(self, script: list, output_dir) -> list:\n        """Build and save each segment as segment_01.mp3, segment_02.mp3, …\n\n        Returns list of Path objects.\n        """\n        parts = self.build(script)\n        out   = Path(output_dir)\n        out.mkdir(parents=True, exist_ok=True)\n        paths = []\n        for i, audio in enumerate(parts, start=1):\n            p = out / f"segment_{i:02d}.mp3"\n            p.write_bytes(audio)\n            paths.append(p)\n        return paths\n'
from pathlib import Path
Path('podcast_generator.py').write_text(_PODCAST_SRC, encoding='utf-8')
print('podcast_generator.py written.')

In [ ]:
import tempfile
from pathlib import Path
from podcast_generator import (
    COMMON_VOICES, DEFAULT_VOICE_MAP,
    build_prosody_ssml, select_voice, synthesize,
    synthesize_segments, PodcastGenerator,
)

mock = lambda text, **kw: b'AUDIO:' + text[:12].encode()

# 1. build_prosody_ssml
ssml = build_prosody_ssml('Hello.', rate='-5%', pitch='-2Hz')
assert '<speak' in ssml and 'rate="-5%"' in ssml and 'Hello.' in ssml
print("\u2705 build_prosody_ssml correct")

# 2. select_voice
v = select_voice(COMMON_VOICES, locale='en-US', gender='Male')
assert v['ShortName'] == 'en-US-GuyNeural'
assert select_voice(COMMON_VOICES, locale='xx-XX') is None
print("\u2705 select_voice correct")

# 3. synthesize
audio = synthesize('Test text.', tts_fn=mock, rate='+10%')
assert isinstance(audio, bytes) and len(audio) > 0
print("\u2705 synthesize correct")

# 4. synthesize_segments
script = [
    {'text': 'Hello.', 'voice': 'en-US-AriaNeural'},
    {'text': 'Hi!',    'voice': 'en-US-GuyNeural', 'rate': '+10%'},
]
parts = synthesize_segments(script, tts_fn=mock)
assert len(parts) == 2 and all(isinstance(p, bytes) for p in parts)
print("\u2705 synthesize_segments correct")

# 5. PodcastGenerator
gen = PodcastGenerator(tts_fn=mock)
podcast = [
    {'text': 'Welcome to the show.',    'voice_key': 'host'},
    {'text': 'Glad to be here!',        'voice_key': 'guest', 'rate': '+10%'},
    {'text': 'And we are underway.',    'voice_key': 'narrator'},
]
built = gen.build(podcast)
assert len(built) == 3 and all(isinstance(p, bytes) for p in built)

with tempfile.TemporaryDirectory() as tmpdir:
    paths = gen.save(podcast, tmpdir)
    names = [Path(p).name for p in paths]
    assert names == ['segment_01.mp3', 'segment_02.mp3', 'segment_03.mp3']
    assert all(Path(p).stat().st_size > 0 for p in paths)

print("\u2705 PodcastGenerator correct")
print("\nPodcast Generator complete!")
